In [1]:
import sys
!{sys.executable} -m pip install transformer_lens
!{sys.executable} -m pip install nltk
!{sys.executable} -m pip install hf_transfer

In [2]:
import numpy as np
import torch
from torch.utils.data import DataLoader, IterableDataset, Dataset
import os
from transformer_lens import HookedTransformer
import nltk
import random
from tqdm import tqdm
from transformers import AutoModelForCausalLM
import json

In [ ]:
# ── CONFIG — only change these three lines to switch models ──────────────
TL_MODEL_NAME = "gemma-2-2b"         # TransformerLens model name
HF_MODEL_ID   = "google/gemma-2-2b"  # HuggingFace model ID for fine-tuning
OUTPUT_DIR    = "./gemma2_ft_toy"     # where the FT checkpoint is saved
# For Gemma 3-1b:  TL_MODEL_NAME="gemma-3-1b-pt"  HF_MODEL_ID="google/gemma-3-1b-pt"  OUTPUT_DIR="./gemma3_1b_ft_toy"
# For Gemma 3-4b:  TL_MODEL_NAME="gemma-3-4b-pt"  HF_MODEL_ID="google/gemma-3-4b-pt"  OUTPUT_DIR="./gemma3_4b_ft_toy"
# ─────────────────────────────────────────────────────────────────────────

In [3]:
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")  # set HF_TOKEN env var or paste here
model = HookedTransformer.from_pretrained(TL_MODEL_NAME)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer


In [4]:
def read_relations_jsonl(path: str):
    """
    Read a JSONL file where each line has {"input": ..., "label": ...}.
    Returns a list of dicts.
    """
    records: List[Dict[str, str]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip blank lines
                records.append(json.loads(line))
    return records
    
text_examples = read_relations_jsonl("gemma_toy_dataset_train.jsonl")

In [5]:
def get_subset(text_examples, start_idx, end_idx):
    subset = []
    labels = []
    for rec in text_examples[start_idx: end_idx]:
        subset.append(model.to_tokens(rec["input"], prepend_bos=False).squeeze())
        labels.append(model.to_tokens(rec["label"], prepend_bos=False).squeeze().item())
    subset = torch.stack(subset, dim=0)
    return subset, labels

dataset, labels = get_subset(text_examples, 0, 8000)

In [6]:
def get_text_trainset(dataset, labels):
    train_dataset_text = []
    for idx in range(dataset.shape[0]):
        ex = dataset[idx]
        label = labels[idx]
        ex_text = model.to_string(ex)
        label_text = model.to_string(label)
        train_dataset_text.append({"prompt": ex_text, "label":label_text})
    return train_dataset_text

train_dataset_text = get_text_trainset(dataset, labels)

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch

model_id = HF_MODEL_ID
tok = model.tokenizer
CUE = ""   # or "" if you didn’t use a cue
raw = [ex for ex in train_dataset_text]

class FixedDataset(Dataset):
    def __init__(self, examples):
        self.recs = []
        for ex in examples:
            prompt_ids = tok(ex["prompt"], add_special_tokens=False)["input_ids"]
            label_ids  = tok(ex["label"], add_special_tokens=False)["input_ids"]  # leading space often helps

            # Build input_ids and labels (loss only on the label)
            input_ids = prompt_ids + label_ids
            labels    = [-100] * len(prompt_ids) + label_ids

            # Hard guarantee: all examples must have identical lengths
            self.recs.append({
                "input_ids": input_ids,
                "labels": labels,
            })

        # Sanity: assert fixed length
        lens_inp = {len(r["input_ids"]) for r in self.recs}
        lens_lab = {len(r["labels"]) for r in self.recs}
        assert len(lens_inp) == 1 and lens_inp == lens_lab, f"Lengths vary: {lens_inp=} {lens_lab=}"
        self.seq_len = next(iter(lens_inp))

    def __len__(self): return len(self.recs)
    def __getitem__(self, i): return self.recs[i]

train_ds = FixedDataset(raw[:-max(1, len(raw)//10)] or raw)
val_ds   = FixedDataset(raw[-max(1, len(raw)//10):] or raw[:min(100, len(raw))])


In [8]:
# ---- Collator that DOES NOT pad; just stacks (since lengths are identical) ----
def no_pad_collator(batch):
    input_ids = torch.tensor([ex["input_ids"] for ex in batch], dtype=torch.long)
    labels    = torch.tensor([ex["labels"]    for ex in batch], dtype=torch.long)
    # Optionally build attention_mask of ones
    attention_mask = torch.ones_like(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}




In [9]:
import gc
torch.cuda.empty_cache()
gc.collect()


20

In [10]:
# ---- Model & Trainer ----
auto_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    learning_rate=2e-5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    #evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=40,
    report_to="none",
)

trainer = Trainer(
    model=auto_model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tok,
    data_collator=no_pad_collator,  # <- no padding added
)

# Quick shape sanity check
b = no_pad_collator([train_ds[0], train_ds[1]])
assert b["input_ids"].shape == b["labels"].shape  # [B, T]

trainer.train()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/tmp/ipykernel_2612/794769628.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
40,1.624600
80,1.262400
120,0.434400
160,0.783600
200,0.746600
240,0.303400
280,0.053600
320,0.060600
360,0.048700
400,0.000400


TrainOutput(global_step=900, training_loss=0.25387860691731073, metrics={'train_runtime': 187.9623, 'train_samples_per_second': 38.306, 'train_steps_per_second': 4.788, 'total_flos': 3323448564940800.0, 'train_loss': 0.25387860691731073, 'epoch': 1.0})